## Imports

In [1]:
import sys
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.loaders.load_kaggle_data import load_data
from src.splits.splitting import create_stratified_kfold
from src.preprocessing.preprocessing import split_features_target
from src.preprocessing.preprocessing import get_feature_types
from src.preprocessing.preprocessing import (
    create_one_hot_preprocessor,
    prepare_lightgbm_features
)
from src.feature_engineering.features import apply_features
from src.feature_engineering.features import FEATURES
from src.models.svm import build_svm
from src.submission.submission import create_submission

## Load data

In [2]:
train, test, sample_submission = load_data()

TARGET = "Will_Buy_EV"
ID_COLUMN = "id"

X, y = split_features_target(
    train,
    target=TARGET,
)

X = X.drop(columns=[ID_COLUMN])

X_test = test

print("X columns:")
print(X.columns.tolist())

print("\nX_test columns:")
print(X_test.columns.tolist())

print(f"\nNumber of features: {X.shape[1]}")
print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")

print("\nSame features in X and X_test:", X.columns.equals(X_test.columns))

X columns:
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

X_test columns:
['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']

Number of features: 13
X shape: (668665, 13)
X_test shape: (286571, 14)

Same features in X and X_test: False


## Cross-validation strategy

In [3]:
cv = create_stratified_kfold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

print(cv)

StratifiedKFold(n_splits=3, random_state=42, shuffle=True)


## Hyperparameter search space

In [4]:
param_distributions = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 0.001, 0.01, 0.1],
    "classifier__kernel": ["rbf"],
    "classifier__class_weight": [None, "balanced"],
}

## Feature engineering

In [5]:
all_features = list(FEATURES.keys())

print(f"Number of engineered features: {len(all_features)}")

X_all_features = apply_features(
    X,
    all_features,
)

X_test_all_features = apply_features(
    X_test,
    all_features,
)

Number of engineered features: 24


## Numerical & categorical features

In [6]:
numeric_features, categorical_features = get_feature_types(
    X_all_features
)

print("\nNumber of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print(X_all_features[categorical_features].dtypes)


Number of numeric features: 31
Number of categorical features: 6
Gender                    str
City_Type                 str
Current_Car_Type          str
Home_Charging_Possible    str
Subsidy_Available         str
Range_Anxiety_Level       str
dtype: object


## Build the SVM pipeline

In [7]:
model = build_svm(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    probability=False
)

## Randomized hyperparameter search + k-fold CV

We take a subset of the entire dataset to reduce the computational cost

In [8]:
X_svm, _, y_svm, _ = train_test_split(
    X_all_features,
    y,
    train_size=100_000,
    stratify=y,
    random_state=42,
)

In [9]:
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=5,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=3,
    return_train_score=True,
)

search.fit(X_svm, y_svm)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


c:\Users\Darío\Desktop\KAGGLE\kaggle_ev_adoption\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[CV 1/3] END classifier__C=100, classifier__class_weight=balanced, classifier__gamma=0.001, classifier__kernel=rbf;, score=(train=0.924, test=0.923) total time= 1.6min


c:\Users\Darío\Desktop\KAGGLE\kaggle_ev_adoption\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[CV 2/3] END classifier__C=100, classifier__class_weight=balanced, classifier__gamma=0.001, classifier__kernel=rbf;, score=(train=0.925, test=0.923) total time= 1.6min


...

[CV 2/3] END classifier__C=1, classifier__class_weight=None, classifier__gamma=scale, classifier__kernel=rbf;, score=(train=0.909, test=0.896) total time= 1.5min


c:\Users\Darío\Desktop\KAGGLE\kaggle_ev_adoption\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[CV 3/3] END classifier__C=1, classifier__class_weight=None, classifier__gamma=scale, classifier__kernel=rbf;, score=(train=0.907, test=0.900) total time= 1.5min


c:\Users\Darío\Desktop\KAGGLE\kaggle_ev_adoption\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__C': [0.1, 1, ...], 'classifier__class_weight': [None, 'balanced'], 'classifier__gamma': ['scale', 0.001, ...], 'classifier__kernel': ['rbf']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",3
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not s

In [11]:
results = pd.DataFrame(search.cv_results_)

results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "param_classifier__C",
        "param_classifier__gamma",
        "param_classifier__class_weight",
        "param_classifier__kernel",
    ]
].sort_values("rank_test_score")

,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_classifier__C,param_classifier__gamma,param_classifier__class_weight,param_classifier__kernel
3,1,0.930537,0.000738,0.931047,10,0.001,NaN,rbf
0,2,0.922992,0.000106,0.924406,100,0.001,balanced,rbf
1,3,0.914342,0.000843,0.948502,1,0.1,balanced,rbf
2,4,0.908089,0.001335,0.951803,100,scale,NaN,rbf
4,5,0.898773,0.002307,0.908044,1,scale,NaN,rbf
